# 04 — Tabular & Charts
Compute YoY/QoQ on a tiny quarterly CSV and save a PNG chart.

In [ ]:
from pathlib import Path; import pandas as pd
Path('data/samples').mkdir(parents=True, exist_ok=True)
p = Path('data/samples/beta_revenue_quarterly.csv')
if not p.exists():
    df = pd.DataFrame({'period': pd.period_range('2023Q1', periods=8, freq='Q').astype(str),
                       'revenue': [1.02,1.05,1.10,1.15,1.18,1.20,1.26,1.30]})
    df.to_csv(p, index=False)
p

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from pathlib import Path

def load_table(path):
    df = pd.read_csv(path)
    idx = pd.PeriodIndex(df['period'], freq='Q').to_timestamp('Q')
    df['date'] = pd.to_datetime(idx)
    df['value'] = pd.to_numeric(df['revenue'], errors='coerce')
    return df[['date','value']].dropna().sort_values('date')

def compute_metrics(df):
    s = df.set_index('date')['value'].asfreq('Q')
    out = pd.DataFrame(index=s.index); out['value']=s
    out['pct_change_qoq']=out['value'].pct_change(1)
    out['pct_change_yoy']=out['value'].pct_change(4)
    out['roll_mean']=out['value'].rolling(4, min_periods=2).mean()
    return out.reset_index()

df = load_table(Path('data/samples/beta_revenue_quarterly.csv'))
dfm = compute_metrics(df); dfm.tail(6)

In [ ]:
from pathlib import Path; import matplotlib.pyplot as plt
out = Path('data/outputs'); out.mkdir(parents=True, exist_ok=True)
fig, ax = plt.subplots(figsize=(8,4))
ax.plot(dfm['date'], dfm['value'], label='Revenue')
ax.plot(dfm['date'], dfm['roll_mean'], linestyle='--', label='Rolling mean')
ax2 = ax.twinx()
ax2.bar(dfm['date'], (dfm['pct_change_yoy']*100).fillna(0), alpha=0.25, width=20, label='YoY %')
ax.legend(loc='upper left'); fig.tight_layout()
png = out/'tabular_stats_chart.png'; fig.savefig(png, dpi=144); plt.close(fig)
png